In [33]:
"This IPYNB file is only used as a test, and a tool for deploying the model. The real Code goes in the same file, but "
"with a .py extension rather than a .ipynb"

'with a .py extension rather than a .ipynb'

In [34]:
from pathlib import Path
import pandas as pd

# Try common working directories (notebook folder and workspace root).
candidate_paths = [
    Path("../../../../SOURCES_AND_DATASHEETS/usgs_data_USGS-01646500.csv"),
    Path("backend/SOURCES_AND_DATASHEETS/usgs_data_USGS-01646500.csv"),
]

csv_path = next((p for p in candidate_paths if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError("Could not locate usgs_data_USGS-01646500.csv")
print(f"Using CSV file at: {csv_path}")

df = pd.read_csv(csv_path)
df

Using CSV file at: ..\..\..\..\SOURCES_AND_DATASHEETS\usgs_data_USGS-01646500.csv


,Unnamed: 0,gage_height_ft,streamflow_cfs,dissolved_oxygen_mg_l,pH,specific_conductance_us_cm,temperature_c,turbidity_fnu,precipitation,rain,snowfall,snow_depth,soil_moisture_0_to_1cm,soil_moisture_1_to_3cm,temperature_2m,wind_speed_10m,vapour_pressure_deficit,precip_3hr,precip_24hr,precip_72hr
0,2010-07-06 00:00:00+00:00,2.73,1600.0,NaN,NaN,366.0,29.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2010-07-06 00:15:00+00:00,2.73,1600.0,NaN,NaN,366.0,29.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2010-07-06 00:30:00+00:00,2.73,1600.0,NaN,NaN,366.0,29.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2010-07-06 00:45:00+00:00,2.73,1600.0,NaN,NaN,366.0,29.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2010-07-06 01:00:00+00:00,2.73,1600.0,NaN,NaN,366.0,29.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
626707,2026-07-06 23:00:00+00:00,NaN,NaN,6.6,8.6,356.0,34.9,6.0,1.0,1.0,0.0,0.0,NaN,NaN,24.65,0.648999,0.118415,29.8,50.2,74.800001
626708,2026-07-07 00:00:00+00:00,NaN,NaN,6.6,8.6,356.0,34.9,6.0,1.8,1.8,0.0,0.0,NaN,NaN,24.10,11.074022,0.062441,31.6,51.4,76.600000
626709,2026-07-07 01:00:00+00:00,NaN,NaN,6.6,8.6,356.0,34.9,6.0,0.7,0.7,0.0,0.0,NaN,NaN,23.45,11.225132,0.051792,32.3,51.5,77.300000
626710,2026-07-07 02:00:00+00:00,NaN,NaN,6.6,8.6,356.0,34.9,6.0,1.1,1.1,0.0,0.0,NaN,NaN,22.85,10.137692,0.016814,33.4,52.0,78.400000


In [35]:
# Flood Action Stage: 5 ft
# Minor Flood Stage: 10 ft
# Moderate Flood Stage: 12 ft
# Major Flood Stage: 14 ft
FLOOD_ACTION_STAGE = 5.0
MINOR_FLOOD_STAGE = 10.0
MODERATE_FLOOD_STAGE = 12.0
MAJOR_FLOOD_STAGE = 14.0

In [36]:
#df.hist(figsize=(10, 6))
# get the instances where Gage Height is > 5
df = df.dropna(subset=['gage_height_ft'])
df_flood = df[df['gage_height_ft'] > FLOOD_ACTION_STAGE]
df_minor_flood = df[df['gage_height_ft'] > MINOR_FLOOD_STAGE]
df_moderate_flood = df[df['gage_height_ft'] > MODERATE_FLOOD_STAGE]
df_major_flood = df[df['gage_height_ft'] > MAJOR_FLOOD_STAGE]

# print the lengths of each of the dataframes
print(f"Total records: {len(df)}")
print(f"Flood Action Stage records: {len(df_flood)}")
print(f"Minor flood records: {len(df_minor_flood)}")
print(f"Moderate flood records: {len(df_moderate_flood)}")
print(f"Major flood records: {len(df_major_flood)}")

Total records: 624474
Flood Action Stage records: 66631
Minor flood records: 1199
Moderate flood records: 55
Major flood records: 0


In [ ]:
# df should have a datetime column and a binary flag column for action stage
# adjust column names to match your data

# Name the 'Unnamed: 0' column as 'datetime' and convert it to datetime type
df['datetime'] = pd.to_datetime(df['Unnamed: 0'])

df = df.sort_values('datetime').reset_index(drop=True)

# how many hours of gap counts as "the storm ended" (tune this to your data's
# sampling frequency, e.g. 6-12h for hourly gauge data, 24-48h for daily)
GAP_HOURS = 12

# isolate just the flagged (action-stage) rows
flood_rows = df[df['gage_height_ft'] > FLOOD_ACTION_STAGE].copy()

# time since previous flagged reading, saved in column 'gap'
flood_rows['gap'] = flood_rows['datetime'].diff()

# start a new event whenever the gap exceeds threshold (or it's the first row)
flood_rows['new_event'] = (
    flood_rows['gap'].isna() | (flood_rows['gap'] > pd.Timedelta(hours=GAP_HOURS))
)
flood_rows['event_id'] = flood_rows['new_event'].cumsum()

# summarize each event
events = flood_rows.groupby('event_id').agg(
    start=('datetime', 'min'),
    end=('datetime', 'max'),
    n_readings=('datetime', 'count'),
    peak_gage_height=('gage_height_ft', 'max')  # adjust column name as needed
).reset_index(drop=True)

events['duration_hours'] = (events['end'] - events['start']).dt.total_seconds() / 3600

print(f"Total flagged readings: {len(flood_rows)}")
print(f"Independent storm events: {len(events)}")
print(events)


Total flagged readings: 1199
Independent storm events: 11
                       start                       end  n_readings  \
0  2011-03-12 07:45:00+00:00 2011-03-13 06:15:00+00:00          91   
1  2011-04-18 03:45:00+00:00 2011-04-19 10:15:00+00:00         123   
2  2011-05-19 14:15:00+00:00 2011-05-20 23:00:00+00:00         127   
3  2012-10-31 11:30:00+00:00 2012-11-01 02:45:00+00:00          61   
4  2014-05-17 10:00:00+00:00 2014-05-18 17:30:00+00:00         127   
5  2018-06-04 14:00:00+00:00 2018-06-06 00:45:00+00:00         140   
6  2018-09-11 04:30:00+00:00 2018-09-12 06:45:00+00:00         106   
7  2018-09-29 06:30:00+00:00 2018-09-30 14:00:00+00:00         127   
8  2018-12-17 05:00:00+00:00 2018-12-18 11:15:00+00:00         123   
9  2022-05-08 23:15:00+00:00 2022-05-09 10:00:00+00:00          43   
10 2025-05-15 09:30:00+00:00 2025-05-16 18:00:00+00:00         131   

    peak_gage_height  duration_hours  
0              10.87           22.50  
1              11.69   